# Ma Soi — RL (PPO from the BC clone) on Colab

Same experiment as section 6 of `train_bc_local.ipynb` (spec
`docs/superpowers/specs/2026-09-17-rl-ppo-from-bc-design.md`): the stage cells below are
copied from there verbatim. Only the setup differs: Colab paths, Node.js, and a Drive
mirror so a disconnect doesn't lose finished iterations.

## Read first: speed

Most of the time is **TypeScript self-play on CPU** (rollouts + benchmarks), not the
GPU. Free Colab gives ~2 vCPU, so expect it to be **slower than your laptop**
(local: ~6 h per 20-iteration stage; Colab ~1.3 s/game → ~25 min per 3000-game iteration). The win is that your machine stays free. Check
the vCPU count printed in section 0; more cores (paid runtimes) help proportionally.
A GPU is optional — PPO takes seconds either way. **A CPU runtime is fine.**

Colab free limits: a session ends after ~12 h, and an unattended tab can be
disconnected. Every stage cell RESUMES: after a disconnect run section 0 → 1 again,
then re-run the same stage cell. Finished iterations come back from Drive; the
interrupted one restarts from scratch.

## Step A — on your machine: package the code

From the repo root, on the branch you want to train with (only COMMITTED files go in):

```powershell
git archive --format=zip HEAD -o .tmp\repo.zip
```

~20 MB. Upload it to Google Drive as **`MyDrive/masoi/repo.zip`**. Re-do this after
every code change you want Colab to use (the notebook prints the commit it runs).

## Step B — Colab

1. Open this notebook in Colab; runtime type **CPU** is fine (T4 also works).
2. Run section 0 → 1 (≈ 5 min: Node 22 + `npm ci` + build).
3. Run the stage cells in order: 6.0 → 6.1 (must print `PILOT OK`) → 6.2 → … → 6.5.
4. Send the 6.5 block (`VERDICT` + `candidate`) to Claude, and download the candidate
   model from `MyDrive/masoi/rl/<run>/champions/`.

## 0. Code, Node.js, dependencies

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/masoi")   # where repo.zip lives
REPO_ZIP = DRIVE_DIR / "repo.zip"
ROOT = Path("/content/repo")
TRAIN_DIR = ROOT / "ai-training"
DRIVE_RL = DRIVE_DIR / "rl"                        # mirror of finished iterations

if not REPO_ZIP.exists():
    raise SystemExit(f"{REPO_ZIP} not found - run `git archive` (Step A) and upload it")

# Fresh unpack every session: the code must be exactly the zip, never a mix.
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
subprocess.run(["unzip", "-q", str(REPO_ZIP), "-d", str(ROOT)], check=True)
commit = subprocess.run(["unzip", "-z", str(REPO_ZIP)], capture_output=True, text=True).stdout.strip().splitlines()[-1:]
print("code from", REPO_ZIP, "| commit", commit[0] if commit else "?")


def node_major():
    try:
        out = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
        return int(out.lstrip("v").split(".")[0]), out
    except (FileNotFoundError, ValueError):
        return 0, "none"


major, version = node_major()
if major < 22:  # repo needs >= 20.19; install 22 (the version used locally)
    print(f"node {version} -> installing Node 22")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && "
                   "apt-get install -y -qq nodejs > /dev/null", shell=True, check=True)
    major, version = node_major()
print("node", version, "| vCPU", os.cpu_count(), "| python", sys.version.split()[0])

In [ ]:
# npm ci: exact lockfile versions. Takes a few minutes the first time in a session.
done = subprocess.run("npm ci --no-audit --no-fund --loglevel=error && npm run build:deps --silent",
                      shell=True, cwd=ROOT)
if done.returncode != 0:
    raise SystemExit(f"npm ci / build failed (exit {done.returncode}) - read the output above")
import torch
import numpy
print("deps OK | torch", torch.__version__, "| numpy", numpy.__version__,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fine)")

## 1. Helpers + Drive mirror

Same names as the local notebook (`run_logged`, `rl_loop`, `latest_champion`,
`show_state`), so the stage cells below are unchanged copies.

Runs live on the fast local disk `/content/repo/.tmp/rl`. Every 10 minutes and at the
end of each command, finished work is mirrored to `MyDrive/masoi/rl`: `state.json`,
champions, benchmarks, logs, and each iteration listed in `state["done"]` (already
pruned to `model/` + `bench*.json` by `--prune-rollouts`). An unfinished iteration is
NOT mirrored, so after a disconnect it restarts cleanly instead of resuming from
`.done` markers whose data is gone.

In [ ]:
import json
import time

NPM = shutil.which("npm") or "npm"
RL = ROOT / ".tmp" / "rl"
CHAMPION0 = ROOT / "apps" / "server" / "assets" / "models" / "village-bc-0002.weights.json"
NO_NIGHT = "vote,final_vote,hunter_shot"
COMMON = [
    "--temperature", "1", "--lr", "1e-4", "--target-kl", "0.01",
    "--shaping-alpha", "1", "--baseline", "role", "--bench-every", "5",
]
RL_ENV = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"}
SYNC_EVERY_S = 600
BIG = shutil.ignore_patterns("roll-*", "enc", "trajectories.jsonl")
sys.path.insert(0, str(TRAIN_DIR))  # 6.5 imports rl_loop's scoring functions


def sync_to_drive():
    # Mirror FINISHED work only (see the markdown above).
    if not RL.exists():
        return
    DRIVE_RL.mkdir(parents=True, exist_ok=True)
    for item in RL.iterdir():
        if item.is_file():
            shutil.copy2(item, DRIVE_RL / item.name)
            continue
        dest = DRIVE_RL / item.name
        dest.mkdir(exist_ok=True)
        for name in ("bench-champion-0000.json", ".bench-0000.done"):
            if (item / name).exists():
                shutil.copy2(item / name, dest / name)
        if (item / "champions").exists():
            shutil.copytree(item / "champions", dest / "champions", dirs_exist_ok=True)
        # rl_loop may be rewriting state.json right now: mirror it only when it parses,
        # otherwise skip this round (a torn copy on Drive would break the next restore).
        try:
            state_text = (item / "state.json").read_text(encoding="utf8")
            done = json.loads(state_text)["done"]
        except (FileNotFoundError, json.JSONDecodeError, KeyError):
            state_text, done = None, []
        for k in done:
            it = item / f"iter-{k:04d}"
            if it.exists():
                shutil.copytree(it, dest / it.name, ignore=BIG, dirs_exist_ok=True)
        if state_text is not None:  # last: state never names an iteration not yet mirrored
            (dest / "state.json").write_text(state_text, encoding="utf8")


def restore_from_drive():
    if DRIVE_RL.exists():
        shutil.copytree(DRIVE_RL, RL, dirs_exist_ok=True)
        print("restored from Drive:", sorted(p.name for p in DRIVE_RL.iterdir()))


def run_logged(cmd, log, cwd=TRAIN_DIR):
    # Stream child output here and into a log; mirror to Drive periodically and at the end.
    log.parent.mkdir(parents=True, exist_ok=True)
    start = last_sync = time.time()
    with log.open("a", encoding="utf8") as f:
        f.write(f"\n$ {' '.join(map(str, cmd))}\n")
        with subprocess.Popen(
            [str(c) for c in cmd], cwd=str(cwd), env=RL_ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf8", errors="replace",
        ) as proc:
            for line in proc.stdout:
                print(line, end="")
                f.write(line)
                if time.time() - last_sync > SYNC_EVERY_S:
                    f.flush()
                    sync_to_drive()
                    last_sync = time.time()
    sync_to_drive()
    print(f"\n{time.time() - start:.0f}s | exit {proc.returncode} | mirrored to {DRIVE_RL}")
    if proc.returncode != 0:
        raise SystemExit(f"failed (exit {proc.returncode}) - full log: {log}")


def rl_loop(name, champion, side, iterations, games, extra=()):
    out = RL / name
    run_logged(
        [sys.executable, "rl_loop.py", "--champion", champion, "--side", side,
         "--iterations", iterations, "--games", games, "--out", out, *COMMON, *extra],
        RL / f"{name}.log",
    )
    return out


def latest_champion(out):
    files = sorted((out / "champions").glob("champion-*.weights.json"))
    return files[-1], len(files) > 1


def show_state(out):
    state = json.loads((out / "state.json").read_text(encoding="utf8"))
    print(f"{'iter':>4} {'score':>7} {'confirm':>8} {'other':>7} {'imbal':>6}")
    for row in state["scores"]:
        print(f"{row['iteration']:>4} {row['score']:>+7.2f} {row.get('confirmScore', float('nan')):>+8.2f} "
              f"{row.get('otherSide', float('nan')):>+7.2f} {row.get('imbalance', float('nan')):>6.2f}")
    print(f"champion score {state['championScore']:+.2f} | other {state.get('championOther')} "
          f"| imbalance {state.get('championImbalance')}")
    champion, promoted = latest_champion(out)
    print("latest champion:", champion.name, "| PROMOTED" if promoted else "| never promoted")
    return champion, promoted


restore_from_drive()
print("RL dir", RL, "| champion-0", CHAMPION0, "->", "found" if CHAMPION0.exists() else "MISSING")

### 6.0 Prerequisites

Checks the zipped code has the four-decision flag, both gates, the anchor and pruning.

In [ ]:
loop_help = subprocess.run([sys.executable, "rl_loop.py", "--help"], cwd=str(TRAIN_DIR), env=RL_ENV,
                           capture_output=True, text=True, encoding="utf8").stdout
selfplay_help = subprocess.run(["npx", "tsx", "apps/server/scripts/selfplay.ts", "--help"], cwd=str(ROOT),
                               capture_output=True, text=True, encoding="utf8").stdout
wanted = ("--learned-decisions", "--balance-slack", "--other-side-slack", "--anchor-kl", "--prune-rollouts")
missing = [flag for flag in wanted if flag not in loop_help]
if "--learned-decisions" not in selfplay_help:
    missing.append("selfplay --learned-decisions")
if missing:
    raise SystemExit(f"missing {missing} - repo.zip is from an older commit; re-run Step A")
if not CHAMPION0.exists():
    raise SystemExit(f"missing {CHAMPION0}")
print("OK - pipeline has the four-decision flag, both gates, the KL anchor and pruning")

### 6.1 Pilot (~45 min)

Catches pipeline errors before an overnight run. The check cell below must print
`PILOT OK`; otherwise stop and send the log (`.tmp/rl/pilot-v2.log`) to Claude.

In [ ]:
PILOT = rl_loop('pilot-v2', CHAMPION0, 'village', 2, 600, ['--train-decisions', NO_NIGHT])

### 6.2 Village (v3: 20 iterations, ~6 h local / ~9 h Colab)

v2 showed PPO does learn — each 5-iteration block made the village **+0.7 to +1.3**
over the champion — but it could never promote: the loop restarted from the champion
after every rejected benchmark (so gains never added up), and the balance gate blocked
any village gain because the village already wins > 50 %. v3 (spec D11):

- `--bench-every 10`: 10 iterations accumulate before a benchmark judges them;
- `--balance-slack -1`: no balance gate while training ONE side — whole-table balance
  is checked once, at the end (6.5), after both sides trained.

Still required on BOTH seed sets: score ≥ champion + 2, wolves side not down > 1 point.

Reading `show_state`: `score` = village strength vs heuristic, `other` = wolves side,
`imbal` = |village win − 50 %| with the whole table on the model (reported, not gated).

In [ ]:
# Spec D11: side stages accumulate 10 iterations per benchmark; balance is gated in 6.5 only.
SIDE_STAGE = ['--bench-every', '10', '--balance-slack', '-1']
VILLAGE = rl_loop('bc-village-v3', CHAMPION0, 'village', 20, 3000, ['--train-decisions', NO_NIGHT, *SIDE_STAGE])
VILLAGE_CHAMP, village_ok = show_state(VILLAGE)

In [ ]:
# Spec: no promotion -> retry ONCE with --lr 3e-4; still nothing -> stop project A.
if not village_ok:
    VILLAGE = rl_loop('bc-village-v3-lr3', CHAMPION0, 'village', 20, 3000,
                      ['--train-decisions', NO_NIGHT, *SIDE_STAGE, '--lr', '3e-4'])
    VILLAGE_CHAMP, village_ok = show_state(VILLAGE)
if not village_ok:
    raise SystemExit('Village never promoted (also at lr 3e-4). Stop here and send both show_state tables to Claude '
                     '-> next is sub-project B (observation).')
print('village champion:', VILLAGE_CHAMP)

### 6.3 Wolves (v3: 20 iterations, ~6 h local / ~9 h Colab)

Starts from the village champion. `--shaping-decisions vote`: wolves' FINAL_VOTE
shaping signal was measured silent. Gate on the village side is now the "other side".
Same accumulate-then-judge rule as 6.2; training the wolves pulls the whole-table
village win rate back toward 50 %, which 6.5 then checks.

In [ ]:
WOLVES = rl_loop('bc-wolves-v3', VILLAGE_CHAMP, 'wolves', 20, 3000,
                 ['--train-decisions', NO_NIGHT, '--shaping-decisions', 'vote', *SIDE_STAGE])
WOLVES_CHAMP, wolves_ok = show_state(WOLVES)

In [ ]:
if not wolves_ok:
    WOLVES = rl_loop('bc-wolves-v3-lr3', VILLAGE_CHAMP, 'wolves', 20, 3000,
                     ['--train-decisions', NO_NIGHT, '--shaping-decisions', 'vote', *SIDE_STAGE, '--lr', '3e-4'])
    WOLVES_CHAMP, wolves_ok = show_state(WOLVES)
CANDIDATE = WOLVES_CHAMP if wolves_ok else VILLAGE_CHAMP
print('wolves promoted' if wolves_ok else 'wolves never promoted -> candidate stays the village champion; skip 6.4')
print('candidate so far:', CANDIDATE)

### 6.4 Night only (optional, 10 iterations per side, ~3 h local each)

Runs only if BOTH 6.2 and 6.3 promoted. Old ablations showed night gradients flip
sign between seed sets; if neither side promotes here, the night bottleneck is the
observation (sub-project B), not more iterations.

In [ ]:
if village_ok and wolves_ok:
    NIGHT_V = rl_loop('bc-night-village-v3', CANDIDATE, 'village', 10, 3000,
                      ['--train-decisions', 'night', *SIDE_STAGE])
    night_v_champ, night_v_ok = show_state(NIGHT_V)
    if night_v_ok:
        CANDIDATE = night_v_champ
    NIGHT_W = rl_loop('bc-night-wolves-v3', CANDIDATE, 'wolves', 10, 3000,
                      ['--train-decisions', 'night', '--shaping-decisions', 'vote', *SIDE_STAGE])
    night_w_champ, night_w_ok = show_state(NIGHT_W)
    if night_w_ok:
        CANDIDATE = night_w_champ
    if not (night_v_ok or night_w_ok):
        print('night flat on both sides -> note for sub-project B')
else:
    print('skipped (needs both 6.2 and 6.3 promoted)')
print('candidate:', CANDIDATE)

### 6.5 Confirm on fresh seeds + verdict (~40 min)

Both models on seed `confirm-0917` (never used by any iteration), 5 × 300 games,
all five setups including `teacher`. Criteria from the spec:

1. every side that promoted in 6.2–6.4 is ≥ +2 over `village-bc-0002`;
2. no side below −1;
3. whole-table imbalance ≤ 6.8 (current 5.8 + 1) — the ONLY balance gate since v3;
4. zero rule violations.

Send the printed block (verdict + candidate path) to Claude: PASS → packaging as
`village-ppo-0001`; FAIL → report, production keeps `village-bc-0002`.

In [ ]:
BENCH = ['--games', '300', '--repeat', '5', '--seed', 'confirm-0917',
         '--setups', 'baseline,village,wolves,all,teacher',
         '--learned-decisions', 'vote,night,final,hunter']
BASE_JSON = RL / 'confirm-v3-bc0002.json'
CAND_JSON = RL / 'confirm-v3-candidate.json'
if not BASE_JSON.exists():
    run_logged([NPM, 'run', 'ai:benchmark', '--', '--model', CHAMPION0, *BENCH, '--out', BASE_JSON],
               RL / 'confirm.log', cwd=ROOT)
if not CAND_JSON.exists():
    run_logged([NPM, 'run', 'ai:benchmark', '--', '--model', CANDIDATE, *BENCH, '--out', CAND_JSON],
               RL / 'confirm.log', cwd=ROOT)

In [ ]:
from rl_loop import imbalance_of, score_of

promoted_sides = {'village'} | ({'wolves'} if wolves_ok else set())
delta = {side: score_of(CAND_JSON, side) - score_of(BASE_JSON, side) for side in ('village', 'wolves')}
imbalance = imbalance_of(CAND_JSON)
violations = sum(row['violations'] for row in json.loads(CAND_JSON.read_text(encoding='utf8'))['rows'])

checks = {
    'promoted sides >= +2': all(delta[s] >= 2.0 for s in promoted_sides),
    'no side below -1': all(d >= -1.0 for d in delta.values()),
    'imbalance <= 6.8': imbalance <= 6.8,
    'zero violations': violations == 0,
}
print('=' * 60)
for side, d in delta.items():
    print(f'{side:<8} delta vs bc-0002 {d:+.2f}' + ('  (trained)' if side in promoted_sides else ''))
print(f'imbalance {imbalance:.2f} | violations {violations}')
for name, ok in checks.items():
    print(f"  [{'x' if ok else ' '}] {name}")
print('VERDICT:', 'PASS' if all(checks.values()) else 'FAIL')
print('candidate:', CANDIDATE)
print('=' * 60)